<img src="https://github.com/moroneyt/MXB301/raw/main/resources/qutlogo.jpg">

# MXB301 Mathematics of AI
# Lesson 3: Backward pass and automatic differentation (AD)

#### Tim Moroney, 2026

A lesson where we pull together all our work on gradients to write the backward pass for our model; and also learn the capabilities and limitations of automatic differentiation.

# Package management

We start by installing the required packages.  If you're running on Colab this will just download pre-compiled code.  Otherwise be prepared to wait a few minutes for installation the first time you run. Feel free to read ahead while you wait.

In [ ]:
import Pkg
if haskey(ENV, "COLAB_GPU") # check if we're on Colab
  if !isfile("/content/MXB301_2026_01_CPU.tgz") # check if we've already downloaded
    # download precompiled Julia environment for Colab
    run(`gdown https://drive.google.com/uc\?id=1mT9XFadzdfK8CWb5a7BYLUkd2RTi2eZc`)

    # replace Colab's Julia environment with downloaded version
    run(`rm -rf /root/.julia`)
    run(`tar -xzf MXB301_2026_01_CPU.tgz -C /root`)
  end
else
  # For any other machine we install the packages in the usual way
  Pkg.activate(".")
  Pkg.add(["CairoMakie", "CodecZlib", "ColorSchemes", "ComponentArrays", "CondaPkg",
           "DifferentiationInterface", "Distributions", "Downloads", "FiniteDiff", "ForwardDiff",
           "HTTP", "JLD2", "LaTeXStrings", "LinearAlgebra", "Lux", "MKL", "MLUtils", "NNlib",
           "NLSolversBase", "OneHotArrays", "Optim", "PythonCall", "QuadGK", "Random",
           "SpecialFunctions", "Statistics", "StatsBase", "ToeplitzMatrices", "Zygote"])
end

using CairoMakie
using DifferentiationInterface
using LaTeXStrings
using LinearAlgebra
using Lux
using Random
using SpecialFunctions
using Statistics

using ComponentArrays: ComponentVector
using Distributions: Normal, Exponential
using Downloads: download
using ForwardDiff: Dual, partials
using JLD2: jldopen
using MLUtils: DataLoader, rand_like, randn_like
using NLSolversBase: only_fg
using NNlib: softmax, sigmoid, scatter as scattergrad, conv, ∇conv_filter, ∇conv_data, DenseConvDims
using OneHotArrays: onehot, onehotbatch, onecold
using StatsBase: crossentropy, sample, Weights
using ToeplitzMatrices: Toeplitz, Hankel
using QuadGK: quadgk

import CodecZlib
import ColorSchemes
import FiniteDiff
import HTTP
import MKL
import Optim
import Zygote

# Set the random seed for reproducibility
rng = Random.seed!(0)

# SVG format scales properly in web pages and PDFs
CairoMakie.activate!(type = "svg")

# Usual model set-up

Let's set up our character-level language model once again.  We've seen this before, so let's go straight to the code.

In [ ]:
# Load some pre-trained parameters for this model
url = "https://github.com/moroneyt/MXB301/raw/main/resources/CLLM_pretrained.jld2"
paramfile = jldopen(download(url))
p = paramfile["p"]
close(paramfile)

chars = ['a':'z'; ' ']      # we only deal with lowercase text and space
vocab_size = length(chars)  # number of characters in our "vocabulary"

# Mappings between characters and indices (a = 1, b = 2, etc.)
idx_to_char(i) = chars[i]
char_to_idx(c) = findfirst(isequal(c), chars)
string_to_idxs(s) = [char_to_idx(c) for c in s]

# Forward pass
So far, so same.  But today we actually need to make a slight modification to our `forward` function.  Up until now we were only interested in evaluating the model on a particular input -- the _forward_ pass.  But today we are going to also compute gradients of the loss function -- the _backward_ pass.  And for that, we need all the intermediate variables that were computed during the forward pass.

So we'll modify the code so that as well as returning the computed $\hat{y}$, it also returns a cache of all the intermediate variables, for use in the backward pass.

In [ ]:
# Forward pass
function forward(u, p)
    X = p.We[:, u]              # 1. embedding
    v = vec(X)                  # 2. flatten
    z1 = p.W1 * v  + p.b1       # 3. first dense layer
    h1 = tanh.(z1)              # 4. activation
    z2 = p.W2 * h1 + p.b2       # 5. second dense layer
    ŷ = softmax(z2)             # 6. softmax for probabilities

    # Return the output along with a cache of intermediate values
    cache = (; u, X, v, z1, h1, z2)
    return ŷ, cache
end

#
We can give it a test on our usual input string.  Notice we get two outputs. The first output is our usual `ŷ`.  (We'll deal with the second output, the cache, shortly.)

In [ ]:
# Give it a test
u = string_to_idxs("alice said")
ŷ, cache = forward(u, p)
ŷ

# Ground truth

To evaluate the model loss we need to compare its output to the ground truth.  The correct next character is space, represented as a one-hot vector $y$.

In [ ]:
y = onehot(char_to_idx(' '), 1:vocab_size)

# Loss function

The loss function is the cross-entropy as we discussed in the last lesson.  And we can evaluate the loss for the one and only prediction our model has made.

In [ ]:
# calculate the loss for our example
crossentropy(y, ŷ)

# Optimising the loss

Our objective today is to try to drive this loss down as much as possible, by adjusting the model's weights and biases.  Wait, you may say, surely $p$ is a pre-trained set of parameters so it's already as good as it can get?  Well no, not for this _one particular example_ it's not.  It could be better by predicting a probability of exactly 1 for space being the next character.

The actual parameter values were learned by training on many thousands of examples, and trying to do a good job on all of them.  For today, since we're taking things step-by-step, we will be satisfied to train on this one single example.  And for such a tiny data set, it will be easy to drive the loss down to practically zero.  Keep in mind we are doing this for pedagogical clarity -- nobody is really training neural nets on only one training example.

## The cache

From our work in the last lesson, we have derived formulas for the gradient for most steps of this model (great work!).  Let's now apply them one by one to build up all the required gradients of the loss function.

First we need those intermediate values from the forward pass we saved earlier.

In [ ]:
(; u, X, v, z1, h1, z2) = cache  # unpack the cache of intermediate values from the forward pass

## Backward pass

Now we wind our way backwards through the model, propagating the gradient of the loss function from one layer to the previous.

We saw in the exercises from last lesson that it's convenient to roll the softmax and  cross-entropy combination into a single operation and take its gradient.  Specifically, we have
$$
\hat{y} = \textrm{softmax}(z_2)\qquad\textrm{and}\qquad L = \textrm{crossentropy}(y, \hat{y})
$$

and we can skip straight to the gradient of $L$ with respect to $z_2$.  It's simply

$$
\nabla_{z_2} L = \hat{y} - y\,.
$$

Let's record this as a variable.  We'll use the name `∇z2` to mean $\nabla_{z_2} L$.  Remember, _all_ the gradients we ever compute are _of_ $L,$ with respect to different variables.  So we don't need to clutter the names by including `L`, we just emphasise what we are taking the gradient _with respect to_.

In [ ]:
∇z2 = ŷ - y

## Second dense layer
We're off to a great start!  Working backwards through our model, the next line is the calculation
$$
z_2 = W_2 h_1 + b_2\,.
$$

Last lesson we derived the famous formulas for back propagation through a dense layer such as this.  They are:

$$
\nabla_{W_2} L = \nabla_{z_2} L \ {h_1}^\top, \quad \nabla_{b_2} L = \nabla_{z_2} L \quad \textrm{and} \quad \nabla_{h_1} L = W_2^\top\, \nabla_{z_2} L\,.
$$

Let's give them a whirl!  Notice that each time, the gradient will be the same size as its corresponding parameter or variable ($W_2, b_2$ and $h_1$ respectively).


In [ ]:
∇W2 = ∇z2 * h1'  # gradient with respect to weights

In [ ]:
∇b2 = ∇z2  # gradient with respect to biases

In [ ]:
∇h1 = p.W2' * ∇z2  # gradient with respect to data

## Activation function

Moving on, the next line working backwards through the model is the nonlinear activation function.

$$
h_1 = \tanh.(z_1)
$$

In the exercises from last lesson, you derived the rule for propagating the gradient through a scalar function applied element-wise.  The result is

$$
\nabla_{z_1} L = \tanh'.(z_1) \odot \nabla_{h_1} L
$$

Conveniently $\tanh'(z) = 1 - \tanh(z)^2$, so we have simply
$$
\nabla_{z_1} L = (1 - {h_1}^{\circ 2}) \odot \nabla_{h_1} L
$$

Let's compute it now.

In [ ]:
∇z1 = (1 .- h1.^2) .* ∇h1

## First dense layer

Now it's on to the first dense layer

$$
z_1 = W_1 v + b_1\,.
$$

We're familiar with this process having done it once already.  We require

$$
\nabla_{W_1} L = \nabla_{z_1} L \ {v}^\top, \quad \nabla_{b_1} L = \nabla_{z_1} L \quad \textrm{and} \quad \nabla_{v} L = W_1^\top\, \nabla_{z_1} L\,.
$$







In [ ]:
∇W1 = ∇z1 * v'  # weights

In [ ]:
∇b1 = ∇z1  # biases

In [ ]:
∇v = p.W1' * ∇z1  # data

## Flattening layer

Nearly done!  We just have two more lines of the model code to traverse.  The next operation is

$$
v = \textrm{vec}(X)
$$

which you also tackled in the exercises from last lesson.  The result is

$$
\nabla_{X} L = \textrm{unvec}(\nabla_{v} L)
$$

where "unvec" just means reshaping back into its rightful matrix shape.

In [ ]:
∇X = reshape(∇v, size(X))

## Embedding layer

Alright, we've made it to the very first line of the model code now.  But what do we have here -- an indexing operation?

$$
X = W_e[:, u]
$$

How are we supposed to differentiate an indexing operation?  What does that even _mean_?

Fortunately we can still reach for our trusty matrix calculus on this one. We just need to think laterally for a moment.  Since $u$ is a vector of column indices, we can reformulate this indexing as the _product_

$$
X = W_e\ R
$$

where $R$ is a **one-hot batch** matrix corresponding to the indices.

Each column of $R$ is "hot" in exactly the corresponding index value in $u$.  For example, $u_1 = 1$ (representing `'a'` in `"alice said"`), and the first column of $R$ has a one in position 1 and zeros elsewhere. Likewise $u_2 = 12$ (representing `'l'`), and the second column of $R$ has a one in position 12 and zeros elsewhere.

In [ ]:
R = onehotbatch(u, 1:vocab_size)

##
The upshot is that _right multiplying_ by $R$ is the same as _indexing_ with $u$.  Confirming, first here's indexing:

In [ ]:
p.We[:, u]  # indexing with u

##
and here's right multiplying by $R$

In [ ]:
p.We * R  # right multiplying by R

## Gradient of embedding layer using $R$

So we actually have the equation
$$
X = W_e R
$$
to propagate through, which is well within our capabilities.  The result (which you should check!) is

$$
\nabla_{W_e} L = \nabla_{X} L\ \, R^\mathrm{T}\,.
$$

We'll calculate it using this formula to begin with.

You can see that many of the entries in the gradient are zero, which makes sense.  Much of the embedding matrix is going unused in our little example, because the only characters in the input are from the text `"alice said"`.  The columns of $W_e$ corresponding to `'b'`, or to `'z'`, or any other character not in the input text could be anything at all, and it wouldn't affect the output of the model for this particular input.

In [ ]:
∇We_onehot = ∇X * R'

## Gradient of embedding layer using $u$
So, we have derived the correct formula for the gradient using the matrix calculus approach, which necessitated forming the one-hot matrix $R$.  Can we now reformulate this calculation to use $u$ directly?

Yes, and it's exactly the kind of thing that libraries provide specialised utilities to do efficiently.  We will be happy to use this functionality rather than muck around with the indexing ourselves.  We've understood the principle.  If you're interested, the reference implementation is described [here](https://fluxml.ai/NNlib.jl/stable/reference/#Gather-and-Scatter).

The operation that undoes the indexing is called "scattering".


In [ ]:
∇We = scattergrad(+, ∇X, u)     # use the scattergrad function to get the gradient
maximum(abs, ∇We - ∇We_onehot)  # check we got the same result as before

# Backward pass code

Putting our carefully-derived gradient rules all down in one function, we arrive at the backward pass for our model.

In [ ]:
# Backward pass: calculate gradient
function backward(p, y, ŷ; cache)

    # gradient vector to fill in
    g = zero(p)

    # unpack the cache of intermediate values from the forward pass
    (; u, X, v, z1, h1, z2) = cache

    # Calculate the gradient of the loss with respect to the model parameters.
    # Each rule is simple enough, but take care!
    ∇z2 = ŷ - y                         # 6. softmax with crossentropy loss
    g.W2 = ∇z2 * h1'                    # 5. second dense layer (wrt weights)
    g.b2 = ∇z2                          # 5. second dense layer (wrt bias)
    ∇h1 = p.W2' * ∇z2                   # 5. second dense layer (wrt data)
    ∇z1 = (1 .- h1.^2) .* ∇h1           # 4. activation
    g.W1 = ∇z1 * v'                     # 3. first dense layer (wrt weights)
    g.b1 = ∇z1                          # 3. first dense layer (wrt bias)
    ∇v = p.W1' * ∇z1                    # 3. first dense layer (wrt data)
    ∇X = reshape(∇v, size(X))           # 2. (un)flatten
    g.We = scattergrad(+, ∇X, u)        # 1. embedding (wrt weights)

    return g
end

#
We can give it a try and confirm we get the same results as before.



In [ ]:
# Backward pass code
g = backward(p, y, ŷ; cache)

# Compare each component with the gradients we already calculated
@show maximum(abs, g.W1 - ∇W1);
@show maximum(abs, g.b1 - ∇b1);
@show maximum(abs, g.W2 - ∇W2);
@show maximum(abs, g.b2 - ∇b2);
@show maximum(abs, g.We - ∇We);

# Finite difference check

We might also like to test whether we have derived and coded the matrix calculus formulas correctly by using _finite differences_ to approximate the gradients.  We did exactly this in the last lesson to verify our gradients with respect to parameters such as $W_1$ and $b_1$.

Now remember, in deriving elegant formulas for these gradients using matrix calculus techniques it was essential to acknowledge that $W_1$ is a matrix, and $b_1$ is a vector.  The formulas depend on this.

But if we just want finite difference approximations, then it's actually completely fine to treat the _entire_ parameter vector $p$ as just one big array of numbers.  That's the beauty of the `ComponentArray` formulation: you _can_ pull out individual components like `p.W1` and `p.b1` if you like.  Or, you can just treat `p` itself as a single huge vector.

As far as finite differences are concerned, we can completely ignore any of the details about the components of $p$, and just treat it as a huge vector in $\mathbb{R}^P$, where $P$ is the total number of learnable parameters in the model.


In [ ]:
P = length(p)   # the total number of parameters in the model

# Loss function
Let's define a function $L(p)$, representing the loss function for the parameter vector $p$.

The derivative of $L$ with respect to the $i$th entry of $p$ is then simply

$$
\frac{\partial L}{\partial p_i} \approx \frac{L(p + \varepsilon e_i) - L(p)}{\varepsilon}\,.
$$

So we define the function $L$, the unit shift vector $e_i$ and a suitably small value of $\varepsilon$:

In [ ]:
# Loss function
L(p) = crossentropy(y, forward(u, p)[1])  # ignoring the returned cache from forward(u, p)

# Unit shift vector
e(i0) = [i == i0 for i = 1:P]  # 1 in position i0

# Small shift value (square root of machine epsilon)
ε = sqrt(eps())

# Full finite difference gradient
Now we can compute the full finite difference gradient.  Notice that this truly is just a big vector in $\mathbb{R}^P$ -- there is no grouping of the values into the components $W_1$, $b_1$, etc.


In [ ]:
g_fd = [(L(p + ε*e(i)) - L(p)) / ε for i = 1:P]

# Using a finite difference package

If you didn't want to code up the finite difference gradient yourself, as we just did, you can use the `gradient` command from the `DifferentiationInterface` library.  Here we get essentially the same result as above without needing to write the boilerplate code.

In [ ]:
# Compute the gradient ∇L(p) using finite differences
g_fd2 = gradient(L, AutoFiniteDiff(), p)

# Confirm both finite difference results agree with the hand-calculated gradient to O(ε)
@show maximum(abs, g - g_fd);
@show maximum(abs, g - g_fd2);

# Forward Mode AD and Dual Numbers

The idea of using finite differences to approximate derivatives can in fact be improved.  This leads to the concept of **dual numbers** and **automatic differentation (AD)**.

$$
\require{cancel}
$$

A **dual number** is a number of the form

$$
a + b \varepsilon,\qquad a,b \in \mathbb{R}
$$

where $\varepsilon$ has the defining property

$$
\varepsilon^2 = 0\,.
$$

n.b. $\varepsilon$ is not zero!  The dual number system is an _extension_ of the real numbers obtained by adjoining this new element $\varepsilon$.  You can see the direct analogies with the _complex number system_ $\mathbb{C}$, which has the new element denoted $i$ and satisfying $i^2 = -1$.

The motivation for the rule $\varepsilon^2 = 0$ is that it represents an infinitesimal value.  We say that $a$ is the _primal_ part, and $b$ is the _dual_ part.  The primal part represents the value, and the dual part represents a first-order infinitesimal variation.

Addition, subtraction, multiplication and division of dual numbers follow from distributivity and the rule $\varepsilon^2 = 0$.  These simple operations naturally re-derive the derivative rules you are familiar with from calculus.


## Addition and subtraction

Addition and subtraction of dual numbers satisfy
$$
(a + b \varepsilon) \pm (c + d \varepsilon) = (a \pm c) + (b \pm d) \varepsilon
$$

The primal part is the sum/difference of the primals, and the dual part shows that derivatives simply add/subtract as well.  This corresponds to the calculus rule

$$
\frac{\textrm{d}}{\textrm{d}t} \left(u(t) \pm v(t)\right) = u'(t) \pm v'(t)
$$

## Multiplication

Multiplication of dual numbers satisfies
$$
\begin{align*}
(a + b \varepsilon) (c + d \varepsilon) &= ab + ad\varepsilon + bc\varepsilon + \xcancel{bd\varepsilon^2}\\
&= ab + (ad + bc)\varepsilon
\end{align*}
$$

and we recognise the product rule
$$
\frac{\textrm{d}}{\textrm{d}t} \left(u(t)\, v(t)\right) = u(t)\,v'(t) + u'(t)\,v(t)\,.
$$

##Function evaluation

For function evaluation $f(a + b\varepsilon)$ we can use the Taylor expansion of $f$ to derive

$$
f(a + b\varepsilon) = f(a) + f'(a) b \varepsilon + \xcancel{\textrm{[higher order terms]}}
$$

and we recognise the chain rule

$$
f'(u(t)) = f'(u)\,u'(t)\,.
$$

## Numerical example 1

So arithmetic of dual numbers gives derivatives for free.  And if the basic primitives like $\sin$, $\cos$, $\exp$, etc. are also defined for dual numbers, you will automatically get derivatives of any composition of these.

For example, let's construct a dual number $u = 2 + \varepsilon$.  We can compute, for example, $\log(u)$.  The primal part of the result is the value $\log(2) \approx 0.6391$ as expected, and the dual part is $0.5$ corresponding to the _derivative_ $\log'(2) = 1/2$.

In [ ]:
# A dual number representing 2 + ε
𝗎 = Dual(2.0, 1.0)

# Calculations with u yield values and derivatives
𝗏 = log(𝗎)

## Numerical example 2

If we build a little function $f(x) = \log(1 + \cos(x)^2) + \tan(x)/x$ then we can evaluate its value and derivative at any point by passing in a dual number.

We can compare the derivative that's been calculated using dual numbers against one computed using finite differences.  The difference between the two results is $\mathcal{O}(\varepsilon)$, which reflects the accuracy of the _finite difference_ calculation. The calculation with dual numbers is correct to machine precision.

In [ ]:
# Function we want to compute value and derivative of
f(x) = log(1 + cos(x)^2) + tan(x)/x

# The dual number u = 2 + ε
𝗎 = Dual(2.0, 1.0)

# This calculates f(u) and f'(u)
𝗏 = f(𝗎)

# extract the dual part of the result
@show dual_derivative = only(partials(𝗏))

# calculate using finite differences
@show fd_derivative = (f(2.0 + ε) - f(2.0)) / ε

# compare the two
fd_derivative - dual_derivative

##
This example highlights an important conceptual difference between finite differences and dual numbers.  Both are used to calculate derivatives, but the former is only approximately correct, with an accuracy that can approach the square root of machine precision.  (Using $\varepsilon = \sqrt{\epsilon_\textrm{mach}}$ is the sweet spot: set $\varepsilon$ any lower and you'll be dominated by truncation error; set $\varepsilon$ any higher and you'll be dominated by cancellation error.)

Dual numbers in contrast are effectively encoding the "finite difference shift" using a _whole extra component_: the dual part of the number.  So you get full precision in both the primal and dual parts.

What's more, you can extend the idea of dual numbers to include multiple dual components, e.g. define a dual number with two independent infinitesimal directions

$$
a + b \varepsilon_1 + c \varepsilon_2
$$

with $\varepsilon_1^2 = \varepsilon_2^2 = \varepsilon_1\varepsilon_2 = 0$, where now you can carry the primal value, and derivatives with respect to _two_ separate variables.  Now you can get partial derivatives $\partial f/\partial s$ and $\partial f/\partial t$ of $f(s, t)$ by evaluating $f$ with $s = s_0 + 1\epsilon_1 + 0\epsilon_2$ and $t = t_0 + 0\epsilon_1 + 1\epsilon_2$.  Continuing in this way, you can carry derivatives with respect to arbitrarily many variables; you just pay the price of one extra dual component per independent variable.

The `Dual` type supports this, which explains the `partials` function we used to extract the dual part earlier.  In general, there can be as many partials (i.e. dual components) as you like.

The cost of operating on a dual number scales with the number of dual components, so essentially if you want derivatives of $f$ with respect to $n$ variables, the cost of evaluating $f$ with dual numbers is $n$-times more than evaluating $f$ with an ordinary number.

(In practice the performance can be significantly worse than this, if there happened to be a very specialised implementation of $f$ for floating point inputs, but only a generic, less-optimised version for generic inputs.  Highly-optimised linear algebra routines are a classic example, where performance for dual inputs can be orders of magnitude worse.)

## Forward Mode AD

Dual numbers is the standard way to implement what is called **Forward Mode Automatic Differentiation** or simply **Forward Mode AD**. It's called _Forward_ mode because it operates on the forward pass through a model code.  (There is no backward pass required.)

If we consider our CLLM code, forward mode AD can differentiate using just the `forward` function.  In this way it is rather analogous to how we used finite differences earlier: for each of the $P$ variables that we want derivatives with respect to, it shifts them independently.  For finite differences, that's done using $P$ floating point shifts.  For forward mode AD, it's done using $P$ dual components.  Either way, we are effectively evaluating the `forward` code $P$ times in order to get the $P$ derivatives we need.

Speaking of which, let's calculate the gradient of our loss function this time using dual numbers.  Just modify the call to `gradient` from earlier, swapping out `AutoFiniteDiff` for `AutoForwardDiff`.

We see the result agrees with our hand-calculated gradient to machine precision.

In [ ]:
# Compute the gradient ∇L(p) using forward mode AD (i.e. dual numbers)
g_forward = gradient(L, AutoForwardDiff(), p)

# Compare to our hand-calculated gradient
maximum(abs, g - g_forward)

## Reverse mode AD

There is also **Reverse Mode Automatic Differentiation**, or **Reverse mode AD**.  As you can guess from the name, reverse mode AD operates on the reverse (or as we called it, `backward`) pass through the model code.

Reverse mode AD involves a very complicated translation of the code in question.  Effectively it tries to automatically write a `backward` function for any arbitrary `forward` code.  We did this ourselves manually for our loss function, using the rules of matrix calculus.  A reverse mode AD system has to have all the rules we used, and many more, as part of its design so that it can "automatically" generate the correct backward pass code from the forward pass code.  And it has to manage all of the bookkeeping, making sure all the intermediate variables are saved from the forward pass for re-use in the backward pass.  And it may have to handle loops, and conditionals, and variables being modified, and so on.  It's very complicated.  As a result there are many different ideas for how to do it, and there is no one right way that has emerged so far, that can handle any arbitrary code you might throw at it.

##
But the payoff, if it can be done, is tremendous.  Rather than requiring $P$ evaluations of the `forward` code to get the $P$ partial derivatives, you just need _one_ pass through the `backward` code.

From this perspective, _back propagation_ -- the algorithm that makes deep learning possible -- is just a particular realisation of the general principle of reverse mode AD.

There are many reverse mode backends we could use to compare against, all with their own strengths and weaknesses, but we'll opt for one that was designed particularly with deep learning in mind, called (for some reason) `Zygote`.  The first time the `gradient` line of code below is run, all of the complex machinery of the reverse mode AD system needs to be compiled.  This can take quite a bit of time up-front.  But once it's done, it has synthesised a fast backward pass of the code (something that took us several weeks of study to do by hand!), so subsequent calls to `gradient` should be fast.

Here we get exact agreement with our hand-coded gradients -- they are computing precisely the same thing.

In [ ]:
# Compute the gradient ∇L(p) using reverse mode AD
g_backward = gradient(L, AutoZygote(), p)

# Compare to our hand-calculated gradient
maximum(abs, g - g_backward)

# Forward vs Reverse mode AD

By its design, Forward mode AD is best suited to taking derivatives with respect to a small number of variables.  The cost is $O(P)$ calls of the forward pass to get derivatives with respect to $P$ variables.  That is, forward mode cost scales with the number of inputs.

In contrast, reverse mode AD scales with the number of _outputs_.  Hence, it is best suited to taking derivatives of a _scalar_ function -- only one output -- with respect to a large number of variables.  The cost is one evaluation of the backward pass code, once it has been synthesised.  But even once the code is synthesised, there is still the complex bookkeeping for all the intermediate variables which is incurred for every call.  So the cross-over point between forward mode and reverse mode efficiency could still be in the tens of variables.

Still, deep learning models involve thousands, millions or even billions of parameters, so forward mode AD is a complete non-starter there.  It's reverse mode or nothing.  If you can code all, or even part, of the backward pass yourself (we managed the whole thing!) you should get the best possible performance.  If you rely on the AD tool to synthesise some or all of the backward pass, the performance can still be very good, but may not be optimal (depending on how good the AD package is of course).

And as for finite difference derivatives, they really have no role to play here.  They are essentially a less-good version of forward mode AD, where numerical stability concerns are lurking around every corner.  If you ever find yourself reaching for finite differences to calculate derivatives, you should seriously consider if it's really the right tool for the job.

# Conclusion

In this lesson we:

* derived formulas for the gradient of the loss function with respect to each model parameter
* learned how to write the backward pass for our model by systematically applying these rules
* introduced dual numbers and forward-mode automatic differentiation (AD)
* discussed reverse mode AD and its advantage for neural nets compared with forward mode AD
* verified our hand-derived gradient formulas using forward and reverse mode AD

In the next lesson we will learn how to efficiently compute the loss and its gradient for multiple training examples, and make a start at actually training the model.